# 1. Онтология

**Нужны:** `mentees.csv`, `mentors.csv`

**Создаёт:** `ontology_domains.csv`, `ontology_role_domain_mapping.csv`, `ontology_domain_similarity.csv`

In [1]:
import csv, math, os
from collections import defaultdict

BASE_DIR = os.path.dirname(os.path.abspath("__file__"))

def load_csv(path):
    with open(path, newline="", encoding="utf-8") as f:
        return list(csv.DictReader(f))

mentees = load_csv(os.path.join(BASE_DIR,"mentees.csv"))
mentors = load_csv(os.path.join(BASE_DIR,"mentors.csv"))
all_profiles = mentees + mentors
print(f"Загружено: {len(mentees)} менти + {len(mentors)} менторов")


Загружено: 5000 менти + 2000 менторов


In [2]:
# Skill→Domain индекс
skill_domain_counts = defaultdict(lambda: defaultdict(int))
for p in all_profiles:
    domain = p.get("domain","").strip()
    if not domain: continue
    for skill in [s.strip().lower() for s in p.get("skills","").split(";") if s.strip()]:
        skill_domain_counts[skill][domain] += 1

skill_domain_index = {}
for skill, dc in skill_domain_counts.items():
    total = sum(dc.values())
    skill_domain_index[skill] = {d: round(c/total,4) for d,c in dc.items()}

domains = sorted({p.get("domain","").strip() for p in all_profiles if p.get("domain","").strip()})
print(f"Областей: {len(domains)}  Навыков: {len(skill_domain_index)}")

# Role→Domain mapping
role_scores = defaultdict(lambda: defaultdict(float))
role_counts = defaultdict(int)
for p in all_profiles:
    prof = p.get("profession","").strip()
    if not prof: continue
    role_counts[prof] += 1
    for skill in [s.strip().lower() for s in p.get("skills","").split(";") if s.strip()]:
        for domain, w in skill_domain_index.get(skill,{}).items():
            role_scores[prof][domain] += w

role_mapping = {}
for prof, ds in role_scores.items():
    total = sum(ds.values())
    role_mapping[prof] = ({d: round(ds.get(d,0)/total,4) for d in domains}
                          if total > 0 else {d: round(1/len(domains),4) for d in domains})

# Матрица похожести областей
all_skills = sorted({s.strip().lower() for p in all_profiles
                     for s in p.get("skills","").split(";") if s.strip()})
dom_freq = defaultdict(lambda: defaultdict(float))
for p in all_profiles:
    d = p.get("domain","").strip()
    if not d: continue
    for s in [x.strip().lower() for x in p.get("skills","").split(";") if x.strip()]:
        dom_freq[d][s] += 1

def vec(d): return [dom_freq[d].get(s,0) for s in all_skills]
def cosine(v1,v2):
    dot = sum(a*b for a,b in zip(v1,v2))
    n1,n2 = math.sqrt(sum(a*a for a in v1)), math.sqrt(sum(b*b for b in v2))
    return round(dot/(n1*n2),4) if n1 and n2 else 0.0

dom_vecs = {d: vec(d) for d in domains}
sim_mat  = {(d1,d2): cosine(dom_vecs[d1],dom_vecs[d2]) for d1 in domains for d2 in domains}

# Сохраняем
with open(os.path.join(BASE_DIR,"ontology_domains.csv"),"w",newline="",encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=["domain_id","domain_name"])
    w.writeheader()
    [w.writerow({"domain_id":f"D{i+1:02d}","domain_name":d}) for i,d in enumerate(domains)]

with open(os.path.join(BASE_DIR,"ontology_role_domain_mapping.csv"),"w",newline="",encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=["profession","n_users"]+domains)
    w.writeheader()
    for prof in sorted(role_mapping):
        row = {"profession":prof,"n_users":role_counts[prof]}
        row.update({d: role_mapping[prof].get(d,0) for d in domains})
        w.writerow(row)

with open(os.path.join(BASE_DIR,"ontology_domain_similarity.csv"),"w",newline="",encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["domain"]+domains)
    [w.writerow([d1]+[sim_mat[(d1,d2)] for d2 in domains]) for d1 in domains]

print(f"ontology_domains.csv             ({len(domains)} областей)")
print(f"ontology_role_domain_mapping.csv ({len(role_mapping)} профессий)")
print(f"ontology_domain_similarity.csv   ({len(domains)}x{len(domains)})")
print("Тетрадка 1 завершена!")


Областей: 10  Навыков: 105
ontology_domains.csv             (10 областей)
ontology_role_domain_mapping.csv (70 профессий)
ontology_domain_similarity.csv   (10x10)
Тетрадка 1 завершена!
